In [1]:
import torch
from dinosaw.linear_head_helpers import get_lin_model, get_val_ds
from dinosaw.utils import seed_everything
from torchmetrics.classification import MulticlassJaccardIndex

/home/pawlo/miniforge3/envs/minimal_dinosaw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED=1025
seed_everything(SEED)

In [ ]:
BENCHMARK="ADE20K"
MODEL="Dv2"
DEVICE="cuda:0"

In [ ]:
ds = get_val_ds(BENCHMARK)
val_dl = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False, drop_last=True)

In [5]:
model = get_lin_model(MODEL, benchmark=BENCHMARK, device=DEVICE)

In [6]:
match BENCHMARK:
    case "VOC12":
        mean_iou = MulticlassJaccardIndex(
            num_classes=21,
            ignore_index=255,
            average="macro",
        ).to(DEVICE)
    case "VOC07":
        mean_iou = MulticlassJaccardIndex(
            num_classes=21, average="macro", ignore_index=255
        ).to(DEVICE)
    case "ADE20K":
        mean_iou = MulticlassJaccardIndex(
            num_classes=150,
            ignore_index=-1,
            average="macro",
        ).to(DEVICE)

loss_fn = torch.nn.CrossEntropyLoss(
                reduction="mean",
                ignore_index=255
                if (BENCHMARK == "VOC12" or BENCHMARK == "VOC07")
                else -1,
            )

In [7]:
def feed_batch_get_loss(
    model,
    loss_fn,
    metric_fn,
    batch: torch.Tensor,
    device: str = "cuda",
):
    x, y_true = batch
    # print(f"{x.shape=}, {y_true.shape=}")
    x = x.to(device, non_blocking=True)
    y_true = y_true.to(device, non_blocking=True)
    model.eval()
    with torch.inference_mode():
        y_pred = model(x)
        loss = loss_fn(y_pred, y_true)
        metric = metric_fn(y_pred, y_true)
    # x = x.to("cpu")
    # y_true = y_true.to("cpu")
    # y_pred = y_pred.to("cpu")
    return loss.detach(), metric.detach()

In [8]:
val_loss_sum, val_miou_sum = 0.0, 0.0
for batch in val_dl:
    loss, miou = feed_batch_get_loss(
        model,
        loss_fn,
        mean_iou,
        batch,
        device=DEVICE,
    )
    val_loss_sum += loss
    val_miou_sum += miou

val_loss = val_loss_sum / len(val_dl)
val_miou = val_miou_sum / len(val_dl)

In [9]:
val_loss, val_miou

(tensor(0.2346, device='cuda:0'), tensor(0.6796, device='cuda:0'))